# Reconstruct MIDI and test samples on a model

## Config

In [27]:
from pathlib import Path
import os
import time
import traceback
import math
import h5py
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import sys
from tqdm.auto import tqdm


In [25]:
# Paths
OUTPUT_PATH = Path(os.path.abspath("")).resolve().parent / "data" / "output"
MODEL_PATH = Path(os.path.abspath("")).resolve().parent / "data" / "models"
DATASET_PATH = Path(os.path.abspath("")).resolve().parent / "data" / "datasets"
# Audio / CQT
SR = 44_100
HOP = 384
FPS = SR / HOP
BINS_PER_OCTAVE = 36
N_OCTAVES = 7
N_BINS = BINS_PER_OCTAVE * N_OCTAVES
FMIN = librosa.note_to_hz("A0")

# MIDI label matrix
MIDI_LO = 21  # A0
MIDI_HI = 108 # C8
N_PITCHES = MIDI_HI - MIDI_LO + 1
CH_ACTIVE = 0
CH_ONSET = 1
CH_VEL = 2
N_LABEL_CHANNELS = 3

# Dataset build
SPLIT = "train"
LIMIT = 10  # Use an integer for smoke tests, or None for the full split.
CHUNK_FRAMES = 384
CHUNK_HOP_FRAMES = 384
ONSET_RADIUS = 1
KEEP_INCOMPLETE = False
MAX_GAP_FRAMES = 43
TARGET_MODE = "active_onset"
TARGET_CHANNELS = ("active", "onset")
N_TARGET_CHANNELS = len(TARGET_CHANNELS)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Load model and dataset

In [35]:
import importlib
import src.models.CNNTransformer as cnntransformer
import src.models.Dataset as dataset
importlib.reload(cnntransformer)
importlib.reload(dataset)

PianoTranscriber = cnntransformer.PianoTranscriber
MaestroChunkDataset = dataset.MaestroChunkDataset
checkpoint = torch.load(MODEL_PATH /"overfit_debug_checkpoint.pt", map_location=DEVICE)

model = PianoTranscriber(
    d_model=checkpoint["config"]["d_model"],
    n_heads=checkpoint["config"]["n_heads"],
    n_layers=checkpoint["config"]["n_layers"],
    max_frames=checkpoint["config"]["chunk_frames"],

).to(DEVICE)
dataset = MaestroChunkDataset(DATASET_PATH / f"maestro_train_chunked.h5")

model.load_state_dict(checkpoint["model_state_dict"])

Loading entire dataset into system RAM... Please wait.
Successfully loaded dataset into RAM!


<All keys matched successfully>

## Define Metrics

In [33]:
def binary_metrics(pred, target, threshold: float, eps: float = 1e-8):
    """
    pred:   numpy array of probabilities, shape (T, 88)
    target: numpy array of 0/1 labels, shape (T, 88)
    """
    pred_bin = pred > threshold
    target_bin = target > 0.5

    tp = (pred_bin & target_bin).sum()
    fp = (pred_bin & ~target_bin).sum()
    fn = (~pred_bin & target_bin).sum()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)

    return {
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def onset_tolerance_metrics(pred_onset, true_onset, threshold=0.5, frame_tolerance=1):
    """
    pred_onset: probabilities, shape (T, 88)
    true_onset: 0/1 labels, shape (T, 88)

    Counts a predicted onset as correct if it matches the same pitch
    within +/- frame_tolerance frames.
    """
    pred_points = np.argwhere(pred_onset > threshold)   # [t, pitch]
    true_points = np.argwhere(true_onset > 0.5)         # [t, pitch]

    matched_true = set()
    tp = 0

    for pred_t, pred_p in pred_points:
        match = None

        for j, (true_t, true_p) in enumerate(true_points):
            if j in matched_true:
                continue

            same_pitch = pred_p == true_p
            close_time = abs(pred_t - true_t) <= frame_tolerance

            if same_pitch and close_time:
                match = j
                break

        if match is not None:
            matched_true.add(match)
            tp += 1

    fp = len(pred_points) - tp
    fn = len(true_points) - len(matched_true)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    return {
        "threshold": threshold,
        "frame_tolerance": frame_tolerance,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def evaluate_one_item_thresholds(
    model,
    dataset,
    idx=0,
    device=DEVICE,
    thresholds=(0.05, 0.1, 0.2, 0.3, 0.5, 0.7),
    onset_frame_tolerance=1,
):
    model.eval()

    X, Y = dataset[idx]          # X: (1, T, 252), Y: (T, 88, 2)
    X_batch = X.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(X_batch)
        probs = torch.sigmoid(logits)[0].cpu().numpy()

    Y = Y.cpu().numpy()

    pred_active = probs[:, :, 0]
    pred_onset = probs[:, :, 1]

    true_active = Y[:, :, 0]
    true_onset = Y[:, :, 1]

    print("Raw probability stats")
    print(f"active: mean={pred_active.mean():.4f}, max={pred_active.max():.4f}")
    print(f"onset : mean={pred_onset.mean():.4f}, max={pred_onset.max():.4f}")
    print()

    print("ACTIVE metrics")
    for th in thresholds:
        print(binary_metrics(pred_active, true_active, th))

    print()
    print("ONSET exact-cell metrics")
    for th in thresholds:
        print(binary_metrics(pred_onset, true_onset, th))

    print()
    print(f"ONSET tolerance metrics, +/- {onset_frame_tolerance} frame")
    for th in thresholds:
        print(onset_tolerance_metrics(
            pred_onset,
            true_onset,
            threshold=th,
            frame_tolerance=onset_frame_tolerance,
        ))

    return {
        "pred_active": pred_active,
        "pred_onset": pred_onset,
        "true_active": true_active,
        "true_onset": true_onset,
    }

## Evaluate

In [49]:
results = evaluate_one_item_thresholds(
    model,
    dataset,
    idx=0,
    device=DEVICE,
    thresholds=(0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99),
    onset_frame_tolerance=3,
)

Raw probability stats
active: mean=0.0295, max=0.9986
onset : mean=0.0019, max=0.9932

ACTIVE metrics
{'threshold': 0.05, 'tp': 618, 'fp': 966, 'fn': 0, 'precision': 0.3901515151490521, 'recall': 0.9999999999838187, 'f1': 0.5613078978645126}
{'threshold': 0.1, 'tp': 618, 'fp': 680, 'fn': 0, 'precision': 0.4761171032320793, 'recall': 0.9999999999838187, 'f1': 0.6450939413433084}
{'threshold': 0.2, 'tp': 618, 'fp': 476, 'fn': 0, 'precision': 0.564899451548767, 'recall': 0.9999999999838187, 'f1': 0.7219626122005198}
{'threshold': 0.3, 'tp': 618, 'fp': 336, 'fn': 0, 'precision': 0.6477987421315744, 'recall': 0.9999999999838187, 'f1': 0.7862595372031546}
{'threshold': 0.5, 'tp': 618, 'fp': 206, 'fn': 0, 'precision': 0.7499999999908981, 'recall': 0.9999999999838187, 'f1': 0.8571428522330098}
{'threshold': 0.7, 'tp': 618, 'fp': 159, 'fn': 0, 'precision': 0.795366795356559, 'recall': 0.9999999999838187, 'f1': 0.8860215004285968}
{'threshold': 0.8, 'tp': 618, 'fp': 140, 'fn': 0, 'precision': 0.